# Indicadores de contexto
> Construir indicadores

In [44]:
from pathlib import Path
import pandas as pd

## Municipios

In [47]:
BASE_DIR = Path.cwd().parent.parent
CARPETA = BASE_DIR / "DATA" / "municipios"
DIVIPOLA = BASE_DIR / "DATA" / "MUN_DIVIPOLA.xlsx"

KEYS = ["codigo_DANE", "nombre_entidad", "anio"]

In [48]:
# DIVIPOLA
divipola = pd.read_excel(DIVIPOLA)

divipola["codigo_DANE"] = (
    divipola["codigo_DANE"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(5)
)

nombre_oficial = dict(
    zip(
        divipola["codigo_DANE"],
        divipola["nombre_entidad"]
    )
)

In [51]:
# Consolidar
df_mun = None

for archivo in Path(CARPETA).glob("*.xlsx"):
    try: 
        df = pd.read_excel(archivo, sheet_name="TABLA")
        df.columns = df.columns.str.strip()

        # Normalizar
        df["codigo_DANE"] = (
            df["codigo_DANE"]
            .astype(str)
            .str.replace(".0", "", regex=False)
            .str.zfill(5)
        )
        df["nombre_entidad"] = df["codigo_DANE"].map(nombre_oficial)
        df = df[[c for c in df.columns]]
        if df_mun is None:
            df_mun = df
        else:
            nuevas = [
                c for c in df.columns 
                if c not in df_mun.columns
                or c in KEYS
            ]
            df_mun = pd.merge(
                df_mun, df[nuevas], 
                on=KEYS, how="outer"
            )
        print(f"📄{archivo.name}")

    except Exception as e:
        print(f"❌{archivo.name}: {e}")

df_mun.to_csv(f"{BASE_DIR / 'DATA' / 'indicadores_municipales.csv'}", index=False, encoding="utf-8-sig")
print(f"✅ Consolidación completa \n {df_mun.shape}")

📄01_Poblacion_total.xlsx
📄02_Poblacion_rural.xlsx
📄03_Poblacion_urbana.xlsx
📄04_NBI_total.xlsx
📄05_Valor_agregado.xlsx
📄06_Peso_relatico_VA.xlsx
📄07_actividades_primarias.xlsx
📄08_actividades_secundarias.xlsx
📄09_actividades_terciarias.xlsx
📄10_Ingresos_tributarios_per_capita.xlsx
📄11_ICLD.xlsx
📄12_Gastos_funcionamiento.xlsx
📄13_Deuda_Publica.xlsx
📄14_Categoria_Ley617.xlsx
📄15_Area_territorial.xlsx
📄16_Cobertura_acueducto.xlsx
📄17_Cobertura_alcantarillado_area_total.xlsx
📄18_ICM.xlsx
✅ Consolidación completa 
 (8992, 27)


## Departamentos